# 🤖 GPT Training on Google Colab (GPU)
**Seedha upload karo aur Run All dabao — bas itna hi!**

Steps:
1. ✅ GPU check
2. ✅ GitHub se code download
3. ✅ PyTorch files replace
4. ✅ Training shuru
5. ✅ Brain save + download

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU mil gaya: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌ GPU nahi mila!')
    print('   Runtime → Change runtime type → T4 GPU → Save karo, phir dobara run karo')

In [ ]:
import os

# 👇 APNA GITHUB REPO URL YAHAN DAALO
GITHUB_REPO = 'https://github.com/niteenmaurya/Colz.git'
FOLDER      = '/content/gpt'

if os.path.exists(FOLDER):
    print('Folder pehle se hai, pull karke latest code aur dataset le raha hoon...')
    os.system(f'cd {FOLDER} && git reset --hard HEAD && git pull')
else:
    os.system(f'git clone {GITHUB_REPO} {FOLDER}')

# Folder ke andar chale jao, ab saara kaam yahi hoga
os.chdir(FOLDER)
print(f'\n📁 Current Workspace Location: {os.getcwd()}')
print('📁 Files available inside repo:')
print('  ' + '\n  '.join(sorted(os.listdir())))

In [ ]:
transformer_code = '''
import torch
import torch.nn as nn
import math

DEVICE = (
    torch.device("cuda")  if torch.cuda.is_available() else
    torch.device("mps")   if torch.backends.mps.is_available() else
    torch.device("cpu")
)

class GPT(nn.Module):
    def __init__(
        self,
        vocab_size:  int,
        d_model:     int,
        n_heads:     int,
        n_layers:    int,
        d_ff:        int,
        max_seq_len: int,
    ) -> None:
        super().__init__()
        self.vocab_size  = vocab_size
        self.d_model     = d_model
        self.max_seq_len = max_seq_len

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb   = nn.Embedding(max_seq_len, d_model)

        self.blocks = nn.ModuleList([
            _TransformerBlock(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])

        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        self.apply(self._init_weights)
        self.to(DEVICE)
        self._last_logits_tensor = None

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_ids) -> list:
        is_list = isinstance(token_ids, list)
        if is_list:
            ids = torch.tensor(token_ids, dtype=torch.long, device=DEVICE)
        else:
            ids = token_ids.to(DEVICE)

        if ids.dim() == 1:
            ids = ids.unsqueeze(0)

        B, T = ids.shape
        if T > self.max_seq_len:
            ids = ids[:, -self.max_seq_len:]
            T = self.max_seq_len

        pos = torch.arange(T, device=DEVICE).unsqueeze(0)
        x = self.token_emb(ids) + self.pos_emb(pos)

        for block in self.blocks:
            x = block(x)

        x      = self.ln_final(x)
        logits = self.lm_head(x)
        self._last_logits_tensor = logits

        if is_list:
            return logits[0].tolist()
        return logits

    def backward(self, dlogits) -> None:
        pass

    def zero_grad(self) -> None:
        for p in self.parameters():
            if p.grad is not None:
                p.grad.detach_()
                p.grad.zero_()

    def all_params_and_grads(self) -> list:
        result = []
        for name, param in self.named_parameters():
            grad = param.grad
            if grad is None:
                grad_list = [[0.0] * param.shape[-1]] * (param.shape[0] if param.dim() > 1 else 1)
            else:
                grad_list = grad.tolist()
            result.append((param.tolist(), grad_list, name))
        return result

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())

class _TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int) -> None:
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = _CausalSelfAttention(d_model, n_heads)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = _FeedForward(d_model, d_ff)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class _CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int) -> None:
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        
        # 🔥 FIX: chunk(3) ensures 3 equal tensors to avoid unpacking crash!
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        scale  = 1.0 / math.sqrt(self.d_head)
        scores = torch.matmul(q, k.transpose(-2, -1)) * scale

        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        scores = scores + mask

        attn = torch.softmax(scores, dim=-1)
        out  = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

class _FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
'''

with open('transformer.py', 'w', encoding='utf-8') as f: 
    f.write(transformer_code)
print('✅ transformer.py with PADDING & CHUNKING fixes generated!')

In [ ]:
trainer_code = '''
import math
import random
import time
import torch
import torch.nn as nn
import torch.optim as optim

DEVICE = (
    torch.device("cuda")  if torch.cuda.is_available() else
    torch.device("mps")   if torch.backends.mps.is_available() else
    torch.device("cpu")
)

class AdamOptimizer:
    def __init__(self, lr=3e-4, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.01):
        self.t = 0
    def step(self, params_and_grads: list) -> None:
        pass

def clip_gradients(params_and_grads: list, max_norm: float = 1.0) -> float:
    return max_norm

def cross_entropy_loss(logits: list, targets: list) -> tuple:
    return 0.0, []

class Trainer:
    def __init__(self, model, lr: float = 3e-4, max_norm: float = 1.0) -> None:
        self.model    = model
        self.max_norm = max_norm
        self.torch_optimizer = optim.AdamW(
            model.parameters(),
            lr=lr,
            betas=(0.9, 0.999),
            eps=1e-8,
            weight_decay=0.01,
        )
        self.optimizer = AdamOptimizer(lr=lr)
        self.loss_history: list = []
        self._loss_fn = nn.CrossEntropyLoss(ignore_index=0)

    def train_step(self, batch_windows: list, seq_len: int) -> float:
        self.torch_optimizer.zero_grad()
        inputs_list, targets_list = [], []
        
        for token_ids in batch_windows:
            if len(token_ids) < 2: continue
            
            inp = token_ids[:-1]
            tgt = token_ids[1:]
            
            if len(inp) > seq_len:
                inp = inp[-seq_len:]
                tgt = tgt[-seq_len:]
                
            pad_len = seq_len - len(inp)
            if pad_len > 0:
                inp = inp + [0] * pad_len
                tgt = tgt + [0] * pad_len
                
            inputs_list.append(inp)
            targets_list.append(tgt)
            
        if not inputs_list: return 0.0

        inp_t = torch.tensor(inputs_list,  dtype=torch.long,  device=DEVICE)
        tgt_t = torch.tensor(targets_list, dtype=torch.long,  device=DEVICE)
        B, T = inp_t.shape

        try:
            with torch.enable_grad():
                pos = torch.arange(T, device=DEVICE).unsqueeze(0)
                x = self.model.token_emb(inp_t) + self.model.pos_emb(pos)
                
                for block in self.model.blocks:
                    x = block(x)
                
                x = self.model.ln_final(x)
                logits = self.model.lm_head(x)

                loss = self._loss_fn(logits.view(-1, logits.size(-1)), tgt_t.view(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_norm)
                self.torch_optimizer.step()
                return loss.item()
        except Exception as e:
            print(f"\\n⚠️ [TRAINER DEBUG] Error in train_step: {str(e)}")
            return 0.0

    def train(self, all_token_ids: list, seq_len: int, epochs: int, log_every: int = 5) -> None:
        windows = [all_token_ids[s : s + seq_len + 1] for s in range(0, len(all_token_ids) - seq_len, seq_len)]
        if not windows:
            windows = [all_token_ids]

        batch_size = 64  
        scheduler = optim.lr_scheduler.CosineAnnealingLR(self.torch_optimizer, T_max=epochs, eta_min=1e-5)

        print("=" * 65)
        print(f"🔥 GPT ENGINE ACCELERATED (DYNAMIC PADDING ACTIVE)!")
        print(f"👉 Target Device : {DEVICE.type.upper()}")
        print(f"👉 Batch Size    : {batch_size} sequences/step")
        print(f"👉 Total Params  : {self.model.count_parameters():,}")
        print(f"👉 Total Windows : {len(windows)}")
        print("=" * 65)
        
        self.model.train()
        start_time = time.time()

        for epoch in range(1, epochs + 1):
            epoch_start_time = time.time()
            random.shuffle(windows)
            batches = [windows[i : i + batch_size] for i in range(0, len(windows), batch_size)]
            
            epoch_loss, steps_run = 0.0, 0
            for b in batches:
                epoch_loss += self.train_step(b, seq_len)
                steps_run += 1
            
            avg_loss = epoch_loss / max(steps_run, 1)
            self.loss_history.append(avg_loss)
            scheduler.step()

            if epoch % log_every == 0 or epoch == 1:
                ppl = math.exp(min(avg_loss, 20))
                lr  = self.torch_optimizer.param_groups[0]['lr']
                elapsed = time.time() - epoch_start_time
                print(f" 🌟 epoch {epoch:4d}/{epochs} | loss={avg_loss:.4f} | ppl={ppl:.2f} | lr={lr:.2e} | time={elapsed:.4f}s")

        print(f"✅ Training completed in {(time.time() - start_time) / 60:.2f} minutes!")
        self.model.eval()
'''

with open('trainer.py', 'w', encoding='utf-8') as f: 
    f.write(trainer_code)
print('✅ trainer.py compiled with high-performance 2D matrix engine!')

In [ ]:
import json
import os

jsonl_file = "dataset.jsonl" 
corpus_text = ""

if os.path.exists(jsonl_file):
    print(f"✅ GitHub repo se '{jsonl_file}' successfully load ho gaya!")
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                for msg in data["messages"]:
                    corpus_text += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
                    
    with open("data.txt", "w", encoding="utf-8") as f:
        f.write(corpus_text)
    print(f"✅ Text conversion complete! Total size: {len(corpus_text):,} characters.")
else:
    print(f"❌ Error: Aapke GitHub repo me '{jsonl_file}' nahi mila!")
    print("💡 Pehle local PC se code aur dataset.jsonl github par push karein, phir isko run karein.")

In [ ]:
import sys
import importlib

sys.path.insert(0, os.getcwd())

for mod in ['tokenizer', 'transformer', 'trainer', 'inference']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from tokenizer  import CharTokenizer
from transformer import GPT
from trainer    import Trainer
from inference  import generate

with open('data.txt', 'r', encoding='utf-8') as f:
    CORPUS = f.read()

tok = CharTokenizer()
tok.build_vocab(CORPUS)
tok.save('vocab.json')
print(f'Vocab size generated: {tok.vocab_size}')
all_ids = tok.encode(CORPUS)

# Optimized network for deep Hinglish dialect mapping
model = GPT(
    vocab_size  = tok.vocab_size,
    d_model     = 256, 
    n_heads     = 8,
    n_layers    = 6,
    d_ff        = 1024,
    max_seq_len = 128,
)
print(f'Model structural parameters: {model.count_parameters():,}')

trainer = Trainer(model, lr=3e-4)
print("\n⚡ Training triggered via GPU acceleration...")
trainer.train(all_ids, seq_len=128, epochs=2000, log_every=5)

In [ ]:
model.eval()

prompts = [
    "<|im_start|>user\nHi!<|im_end|>\n<|im_start|>assistant\n",
    "<|im_start|>user\nKya chal raha hai?<|im_end|>\n<|im_start|>assistant\n"
]

print('=' * 55)
print('🤖 CHATBOT MODEL GENERATION TEST')
print('=' * 55)

for p in prompts:
    out = generate(model, tok, p, max_new=60, strategy='top_k', k=5, temperature=0.7)
    print(f"\nPrompt Input:\n{p}")
    print(f"AI Response:\n{out}")
    print('-'*40)

In [ ]:
import torch
import pickle
from google.colab import files
import matplotlib.pyplot as plt

# 1. PyTorch weights saving
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'vocab_size' : tok.vocab_size,
        'd_model'    : 256,
        'n_heads'    : 8,
        'n_layers'   : 6,
        'd_ff'       : 1024,
        'max_seq_len': 128,
    },
    'loss_history': trainer.loss_history,
}, 'brain.pt')

# 2. Pickle backup saving
with open('brain.pkl', 'wb') as f:
    pickle.dump(model, f)

print('✅ brain.pt saved inside colz repository!')
print('✅ vocab.json created successfully!')

# 3. Loss curve rendering
plt.figure(figsize=(10, 4))
plt.plot(trainer.loss_history, color="crimson", linewidth=1.5)
plt.title("Training Loss Curve", fontsize=14)
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=120)
plt.show()
print("✅ loss_curve.png generated!")

print('\n⬇️  Downloading files directly to your PC... Please wait!')

# 4. Trigger browser multi-download protocols
files.download('brain.pt')
files.download('vocab.json')
files.download('loss_curve.png')

print('🚀 DONE! Files checking complete.')